# Deliverable 3: verify and launch interface v2
This notebook does not retrain any model. It checks the uploaded three-seed results, runs API tests, optionally analyzes incoming graph degree, verifies native inference, and displays the refined interface.

Run in Colab. A CPU runtime is sufficient for this notebook. Keep your existing training notebook and its outputs.


In [ ]:
from google.colab import files
from pathlib import Path
import sys, zipfile, subprocess, json
uploaded = files.upload()  # Choose AML_GNN_Deliverable_3_Prototype.zip
archive_name = next(name for name in uploaded if name.endswith('.zip'))
DEST = Path('/content/aml_deliverable3_ui')
DEST.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_name) as archive:
    for entry in archive.infolist():
        p = Path(entry.filename)
        assert not p.is_absolute() and '..' not in p.parts
    archive.extractall(DEST)
ROOT = DEST / 'aml-gnn'
assert (ROOT / 'ui/app.py').is_file()
sys.path.insert(0, str(ROOT))
print('Project extracted:', ROOT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt'), 'xgboost>=3,<4'], check=True)
subprocess.run([sys.executable, '-m', 'src.refinement_audit'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=ROOT, check=True)
print('PASS: prediction audit and interface API tests.')


## Optional extended diagnostic: where does graph structure help?
This command requires the original CSVs. It groups errors by incoming degree without using neighbor labels as features. It changes no model, threshold or primary metric. AP here pools transactions within a phase/degree group and must not be compared directly with the snapshot-macro AP in the main table. The script has been syntax checked but needs your real-data execution.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks/AML_GNN_Project/data')
assert (DATA_DIR / 'elliptic_txs_features.csv').is_file(), 'Update DATA_DIR to your CSV folder.'
subprocess.run([sys.executable, '-m', 'src.degree_analysis', '--data-dir', str(DATA_DIR)], cwd=ROOT, check=True)


## Verify inference on the real dataset
The reference is the original frozen XGBoost checkpoint. This checks 50 real test rows, the lightweight evaluator, and the actual example served by the new app. No fitting or threshold selection occurs.


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from src.data import load_elliptic
from src.inference import PortableXGBoost
from src import service
nodes, edges = load_elliptic(DATA_DIR)
columns = [f'local_{i:02}' for i in range(93)]
saved = pd.read_csv(ROOT / 'results/xgb_frozen_test_predictions.csv').head(50)
x = nodes.set_index('txId').loc[saved.txId, columns].to_numpy(dtype=np.float32)
booster = xgb.Booster()
booster.load_model(str(ROOT / 'results/xgb_frozen_seed0.json'))
booster.set_param({'device':'cpu', 'nthread':2})
native = booster.predict(xgb.DMatrix(x, feature_names=columns))
portable = PortableXGBoost(ROOT / 'results/xgb_frozen_seed0.json').predict(x)
np.testing.assert_allclose(native, saved.score.to_numpy(), atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(portable, native, atol=1e-5, rtol=1e-5)
example = service.example()
np.testing.assert_allclose(example['features'], x[0], atol=1e-7, rtol=1e-7)
actual = service.live(example['features'])
np.testing.assert_allclose(actual['score'], native[0], atol=1e-5, rtol=1e-5)
verification = {'status':'passed', 'actual_rows':50, 'max_native_vs_saved_difference':float(np.max(np.abs(native-saved.score.to_numpy()))), 'max_portable_vs_native_difference':float(np.max(np.abs(portable-native))), 'live_output':actual}
(ROOT / 'docs/interface_v2_real_verification.json').write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))


## Measure the refined backend in this runtime
This comparison verifies equal outputs and times 30 warm calls per version. It excludes HTTP, browser, network, startup and human interaction. The report must identify the measurement environment.


In [ ]:
subprocess.run([sys.executable, '-m', 'src.refinement_benchmark'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'src.refinement_figures'], cwd=ROOT, check=True)


## Launch interface v2
The app runs in a local Colab process with an iframe, without a public sharing tunnel. If the interface remains open from a previous run of this cell, terminate its process first using `server.terminate()`.


In [ ]:
import time, urllib.request
from google.colab import output
PORT = 8001
log_path = ROOT / 'docs/ui_v2_server.log'
log_handle = log_path.open('w')
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'ui.app:app', '--host', '0.0.0.0', '--port', str(PORT)], cwd=ROOT, stdout=log_handle, stderr=subprocess.STDOUT)
ready = False
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(log_path.read_text())
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/', timeout=1) as response:
            ready = response.status == 200
        if ready: break
    except Exception:
        time.sleep(.25)
assert ready, log_path.read_text()
print('Interface v2 ready. Use the three tabs below.')
output.serve_kernel_port_as_iframe(PORT, height=1100)


## Capture evidence and try the improvements
1. **Compare experiments:** capture the three-seed table and paired gains.
2. **Explore a snapshot:** choose GraphSAGE / recency / seed 1 / step 49 / budget 10. Update, inspect the confusion counts, and download the review queue. Change to step 46 to check the low-support warning.
3. **Predict from features:** load the verified real example and generate a prediction. Download the example CSV, upload it, predict, and download predictions. Capture a screenshot of both results.
4. **Error handling:** remove one number from the JSON array and predict again. Capture the explanatory error. A CSV with a missing column should also produce a clear error.

Record which tasks you completed, any confusing behavior, and whether CSV exports match the displayed selection. Your own walkthrough is a developer test, not an independent user study. If you invite a classmate, record their actual feedback without inventing responses or scores.

Send back the executed notebook, new screenshots, and the evidence ZIP from the next cell. Genuine browser evidence is required before finalizing the IEEE report.


In [ ]:
import shutil
# Download the current evaluated outputs and verification records.
evidence_dir = Path('/content/AML_GNN_D3_New_Evidence')
evidence_dir.mkdir(exist_ok=True)
for name in ['degree_error_analysis.csv', 'transaction_incoming_degree.csv', 'backend_latency.json', 'independent_audit.json']:
    source = ROOT / 'results/deliverable3/evaluation' / name
    if source.is_file(): shutil.copy2(source, evidence_dir / name)
for name in ['interface_v2_real_verification.json', 'ui_v2_server.log']:
    source = ROOT / 'docs' / name
    if source.is_file(): shutil.copy2(source, evidence_dir / name)
archive = shutil.make_archive('/content/AML_GNN_D3_New_Evidence', 'zip', root_dir=evidence_dir)
files.download(archive)
